## Parte 1: Torneios (limpeza e tipagem)
Extração e tipagem dos campos do torneio. Identificados 29 torneios com
o mesmo torneio_uri publicados em datas diferentes (mesmo conteúdo de
decks, confirmado via investigação) — mantida apenas a versão mais
recente de cada.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df_bronze = spark.table("workspace.bronze.mtgo_torneios_raw")

torneios_bruto = df_bronze.select(
    F.col("Tournament.Uri").alias("torneio_uri"),
    F.col("Tournament.Name").alias("nome"),
    F.to_date(F.col("Tournament.Date")).alias("data"),
    "fonte"
)

janela_torneio = Window.partitionBy("torneio_uri").orderBy(F.desc("data"))

torneios = torneios_bruto.withColumn(
    "linha_num", F.row_number().over(janela_torneio)
).filter("linha_num = 1").drop("linha_num")

spark.sql("DROP TABLE IF EXISTS workspace.silver.torneios")
torneios.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.torneios")

torneios.count()

## Parte 2: Decks (explode + parsing de resultado)
Explosão do array Decks e parsing de resultado_bruto em tipo_resultado/
colocacao/vitorias/derrotas. Decisão de qualidade de dados: não foi aplicada
deduplicação por linha completa — o pequeno espaço de valores possíveis em
record_liga (ex: "3-2") faz coincidências entre submissões legítimas do
mesmo jogador/deck plausíveis, e a fonte não oferece campo que distinga
esse cenário de uma duplicata real. AnchorUri identificado como não
confiável como chave natural de deck; usado identificador técnico
(linha_deck) no lugar.

In [0]:
decks_explodido = df_bronze.select(
    F.col("Tournament.Uri").alias("torneio_uri"),
    F.explode("Decks").alias("deck")
).select(
    "torneio_uri",
    F.col("deck.AnchorUri").alias("deck_uri"),
    F.col("deck.Player").alias("nome_jogador"),
    F.col("deck.Result").alias("resultado_bruto"),
    F.col("deck.Mainboard").alias("mainboard"),
    F.col("deck.Sideboard").alias("sideboard")
)

decks_com_resultado = decks_explodido.withColumn(
    "tipo_resultado",
    F.when(F.col("resultado_bruto").rlike(r"^\d+(st|nd|rd|th) Place$"), "colocacao")
     .when(F.col("resultado_bruto").rlike(r"^\d+-\d+$"), "record_liga")
     .otherwise("desconhecido")
).withColumn(
    "colocacao",
    F.when(F.col("tipo_resultado") == "colocacao",
           F.regexp_extract("resultado_bruto", r"^(\d+)", 1).cast("int"))
).withColumn(
    "vitorias",
    F.when(F.col("tipo_resultado") == "record_liga",
           F.regexp_extract("resultado_bruto", r"^(\d+)-\d+$", 1).cast("int"))
).withColumn(
    "derrotas",
    F.when(F.col("tipo_resultado") == "record_liga",
           F.regexp_extract("resultado_bruto", r"^\d+-(\d+)$", 1).cast("int"))
).withColumn(
    "linha_deck", F.monotonically_increasing_id()
)

decks = decks_com_resultado.select(
    "linha_deck", "deck_uri", "torneio_uri", "nome_jogador",
    "tipo_resultado", "colocacao", "vitorias", "derrotas"
)

spark.sql("DROP TABLE IF EXISTS workspace.silver.decks")
decks.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.decks")

decks.count()

## Parte 3: Cartas por deck (explode de mainboard/sideboard)
Explosão dos arrays Mainboard e Sideboard, referenciando cada carta ao
deck via linha_deck.

In [0]:
mainboard_explodido = decks_com_resultado.select(
    "linha_deck", F.explode("mainboard").alias("carta")
).select(
    "linha_deck",
    F.col("carta.CardName").alias("nome_carta"),
    F.col("carta.Count").alias("quantidade"),
    F.lit(False).alias("is_sideboard")
)

sideboard_explodido = decks_com_resultado.select(
    "linha_deck", F.explode("sideboard").alias("carta")
).select(
    "linha_deck",
    F.col("carta.CardName").alias("nome_carta"),
    F.col("carta.Count").alias("quantidade"),
    F.lit(True).alias("is_sideboard")
)

deck_cartas = mainboard_explodido.unionByName(sideboard_explodido)

spark.sql("DROP TABLE IF EXISTS workspace.silver.deck_cartas")
deck_cartas.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.deck_cartas")

deck_cartas.count()

## Parte 4: Enriquecimento de cartas com dados do Scryfall
Padronização de atributos (mana_cost, cmc, type_line, colors) via join com
o bulk data do Scryfall. Tratamento de cartas partidas (nome combinado) e
dupla-face (fallback para card_faces[0] quando o campo principal vem vazio,
aplicado apenas quando card_faces existe de fato). Deduplicação com critério
de desempate priorizando linhas com dado preenchido.

In [0]:
scryfall_bronze = spark.table("workspace.bronze.scryfall_cartas_raw")

scryfall_com_fallback = scryfall_bronze.withColumn(
    "mana_cost_final",
    F.coalesce(
        F.when(F.col("mana_cost") != "", F.col("mana_cost")),
        F.col("card_faces")[0]["mana_cost"]
    )
).withColumn(
    "colors_final",
    F.coalesce(
        F.when(F.size(F.col("colors")) > 0, F.col("colors")),
        F.col("card_faces")[0]["colors"]
    )
)

scryfall_completo = scryfall_com_fallback.select(
    F.col("name").alias("chave_busca"),
    F.col("mana_cost_final").alias("mana_cost"),
    "cmc",
    "type_line",
    F.col("colors_final").alias("colors")
)

scryfall_frente = scryfall_completo.filter(F.col("chave_busca").contains(" // ")).select(
    F.split(F.col("chave_busca"), " // ").getItem(0).alias("chave_busca"),
    "mana_cost", "cmc", "type_line", "colors"
)

scryfall_unificado = scryfall_completo.unionByName(scryfall_frente)

janela_chave = Window.partitionBy("chave_busca").orderBy(
    F.col("mana_cost").isNull().asc(),
    F.col("colors").isNull().asc()
)

scryfall_deduplicado = scryfall_unificado.withColumn(
    "linha_num", F.row_number().over(janela_chave)
).filter("linha_num = 1").drop("linha_num")

In [0]:
cartas_unicas = deck_cartas.select("nome_carta").distinct()

cartas_enriquecidas = cartas_unicas.join(
    scryfall_deduplicado, cartas_unicas.nome_carta == scryfall_deduplicado.chave_busca, how="left"
).drop("chave_busca")

spark.sql("DROP TABLE IF EXISTS workspace.silver.cartas")
cartas_enriquecidas.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.cartas")

cartas_enriquecidas.filter("cmc IS NOT NULL").count()

## Parte 5: Validação visual das tabelas da Silver
Consulta individual das 4 tabelas finais, exibindo as 5 primeiras linhas
de cada, para conferência da estrutura.

In [0]:
%sql
SELECT * FROM workspace.silver.torneios LIMIT 5

In [0]:
%sql
SELECT * FROM workspace.silver.decks LIMIT 5

In [0]:
%sql
SELECT * FROM workspace.silver.deck_cartas LIMIT 5

In [0]:
%sql
SELECT * FROM workspace.silver.cartas LIMIT 5

## Parte 6: Validação de Qualidade de Dados
Verificação de valores nulos nas tabelas finais da Silver. Investigação
detalhada dos nulos em `cartas` (mana_cost/colors), concluindo que 100%
dos casos correspondem a categorias incolores por regra do jogo (terrenos,
artefatos, criaturas Eldrazi, feitiços/planeswalkers colorless).

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN nome IS NULL THEN 1 ELSE 0 END) AS nome_nulo,
    SUM(CASE WHEN data IS NULL THEN 1 ELSE 0 END) AS data_nula,
    SUM(CASE WHEN torneio_uri IS NULL THEN 1 ELSE 0 END) AS uri_nula
FROM workspace.silver.torneios

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN deck_uri IS NULL THEN 1 ELSE 0 END) AS deck_uri_nulo,
    SUM(CASE WHEN nome_jogador IS NULL THEN 1 ELSE 0 END) AS jogador_nulo,
    SUM(CASE WHEN tipo_resultado IS NULL THEN 1 ELSE 0 END) AS tipo_resultado_nulo,
    SUM(CASE WHEN colocacao IS NULL AND tipo_resultado = 'colocacao' THEN 1 ELSE 0 END) AS colocacao_faltando,
    SUM(CASE WHEN vitorias IS NULL AND tipo_resultado = 'record_liga' THEN 1 ELSE 0 END) AS vitorias_faltando,
    SUM(CASE WHEN derrotas IS NULL AND tipo_resultado = 'record_liga' THEN 1 ELSE 0 END) AS derrotas_faltando
FROM workspace.silver.decks

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN mana_cost IS NULL THEN 1 ELSE 0 END) AS mana_cost_nulo,
    SUM(CASE WHEN cmc IS NULL THEN 1 ELSE 0 END) AS cmc_nulo,
    SUM(CASE WHEN type_line IS NULL THEN 1 ELSE 0 END) AS type_line_nulo,
    SUM(CASE WHEN colors IS NULL THEN 1 ELSE 0 END) AS colors_nulo
FROM workspace.silver.cartas

### Investigação dos nulos em `cartas`

In [0]:
%sql
SELECT nome_carta, mana_cost, cmc, type_line, colors
FROM workspace.silver.cartas
WHERE colors IS NULL
LIMIT 30

In [0]:
spark.table("workspace.silver.cartas") \
    .filter("colors IS NULL") \
    .groupBy("type_line").count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

In [0]:
%sql
-- Confirma que a maioria dos nulos é artefato ou terreno (incolores por regra)
SELECT COUNT(*) FROM workspace.silver.cartas
WHERE colors IS NULL AND (type_line LIKE '%Artifact%' OR type_line LIKE '%Land%')

In [0]:
%sql
-- Investiga o restante: deveriam ser Eldrazi e feitiços/planeswalkers colorless
SELECT nome_carta, type_line, cmc, colors
FROM workspace.silver.cartas
WHERE colors IS NULL 
  AND type_line NOT LIKE '%Artifact%' 
  AND type_line NOT LIKE '%Land%'
LIMIT 119

In [0]:
spark.table("workspace.silver.cartas") \
    .filter("colors IS NULL") \
    .groupBy("type_line").count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

In [0]:
%sql
SELECT nome_carta, type_line, cmc, colors
FROM workspace.silver.cartas
WHERE colors IS NULL 
  AND type_line NOT LIKE '%Artifact%' 
  AND type_line NOT LIKE '%Land%'
LIMIT 30

### Conclusão
100% dos valores nulos em `mana_cost` (574) e `colors` (1.444) correspondem
a categorias incolores por regra do Magic: terrenos, artefatos, criaturas
Eldrazi, e feitiços/planeswalkers "colorless" (ex: Karn Liberated, Void
Shatter). Nenhum caso de nulo sem explicação de domínio foi identificado
em `torneios`, `decks` ou `cartas`.